# Teaching transformers how to write like Shakespeare

### Imports

In [1]:
import os
import sys
import numpy as np
import torch 
from tqdm import tqdm
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve() / "src"))
from transformers.word_embedding import word_embedding
from transformers.utils import estimate_loss, get_batch
from transformers.transformer import GPTLanguageModel

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/juanfrancisco/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/juanfrancisco/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
input_file_path = "/Users/juanfrancisco/Desktop/Transformers/data/tinyshakespeare/"

# # download the tiny shakespeare dataset
# if not os.path.exists(input_file_path + "input.txt"):
#     data_url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
#     with open(input_file_path + "input.txt", 'w', encoding='utf-8') as f:
#         f.write(requests.get(data_url).text)

with open(input_file_path + "input.txt", 'r', encoding='utf-8') as f:
    text = f.read()

print(text[:100])
print(f"length of dataset in characters: {len(text)}")

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You
length of dataset in characters: 1115394


### Word embedding

In [11]:
text = text[:500]

embedding_model = word_embedding(embedding_type="gpt2")
tokens = embedding_model.encode(text)
n_vocab = embedding_model.tokenizer.n_vocab
embedding = embedding_model.embed(text)
# embedding = torch.nn.Embedding(n_vocab, n_vocab)
# embedding.load_state_dict(torch.load(os.path.join(input_file_path, 'embedding.pt')))
data = torch.tensor(tokens, dtype=torch.long)

# Split training and validation dataset
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [12]:
print(f"train has {len(train_data):,} tokens")
print(f"val has {len(val_data):,} tokens")

train has 140 tokens
val has 16 tokens


In [13]:
# # export files
# train_ids = np.array(train_data, dtype=np.uint16)
# val_ids = np.array(val_data, dtype=np.uint16)
# train_ids.tofile(os.path.join(input_file_path, 'train.bin'))
# val_ids.tofile(os.path.join(input_file_path, 'val.bin'))
# torch.save(embedding.state_dict(), os.path.join(input_file_path, 'embedding.pt'))

Split data

In [14]:
block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([5962]) the target: 22307
when input is tensor([ 5962, 22307]) the target: 25
when input is tensor([ 5962, 22307,    25]) the target: 198
when input is tensor([ 5962, 22307,    25,   198]) the target: 8421
when input is tensor([ 5962, 22307,    25,   198,  8421]) the target: 356
when input is tensor([ 5962, 22307,    25,   198,  8421,   356]) the target: 5120
when input is tensor([ 5962, 22307,    25,   198,  8421,   356,  5120]) the target: 597
when input is tensor([ 5962, 22307,    25,   198,  8421,   356,  5120,   597]) the target: 2252


In [15]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

xb, yb = get_batch('train', train_data, val_data, block_size, batch_size)
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

inputs:
torch.Size([4, 8])
tensor([[  389,   477, 12939,  2138,   284,  4656,   621,   284],
        [  621,   284,  1145,   680,    30,   198,   198,  3237],
        [  198,  5756,   514,  1494,   683,    11,   290,   356],
        [  385,  1526, 28599,   318,  4039,  4472,   284,   262]])
targets:
torch.Size([4, 8])
tensor([[  477, 12939,  2138,   284,  4656,   621,   284,  1145],
        [  284,  1145,   680,    30,   198,   198,  3237,    25],
        [ 5756,   514,  1494,   683,    11,   290,   356,  1183],
        [ 1526, 28599,   318,  4039,  4472,   284,   262,   661]])


### Training

In [16]:
vocab_size = embedding.num_embeddings
vocab_dim = 32
epochs = 10

eval_iters = 10
head_size = 8
num_heads = vocab_dim // head_size

# create a PyTorch optimizer
model = GPTLanguageModel(vocab_size, vocab_dim, block_size, num_heads)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in tqdm(range(epochs)): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train', train_data, val_data, block_size, batch_size)

    losses = estimate_loss(model, train_data, val_data, block_size, batch_size, eval_iters)
    print(f"step {step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)

    #update parameters
    loss.backward()
    optimizer.step()

 10%|█         | 1/10 [00:00<00:01,  5.82it/s]

step 0: train loss 10.9613, val loss 11.0825
step 1: train loss 10.9060, val loss 11.0460


 70%|███████   | 7/10 [00:00<00:00, 19.17it/s]

step 2: train loss 10.8682, val loss 11.0054
step 3: train loss 10.8620, val loss 10.9906
step 4: train loss 10.8313, val loss 10.9476
step 5: train loss 10.8509, val loss 10.9240
step 6: train loss 10.7934, val loss 10.8904
step 7: train loss 10.7817, val loss 10.8690


100%|██████████| 10/10 [00:00<00:00, 19.12it/s]

step 8: train loss 10.7523, val loss 10.8407
step 9: train loss 10.7256, val loss 10.8146
